In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
base_path = "/content/drive/MyDrive/textile_dataset"

In [6]:
import cv2
import os

def process_folder(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    for img_name in os.listdir(input_folder):
        img_path = os.path.join(input_folder, img_name)
        img = cv2.imread(img_path)

        if img is None:
            continue

        img = cv2.resize(img, (64, 64))
        cv2.imwrite(os.path.join(output_folder, img_name), img)

# Create processed dataset
process_folder(base_path + "/train/normal", "/content/processed/train/normal")
process_folder(base_path + "/train/defect", "/content/processed/train/defect")
process_folder(base_path + "/test/normal", "/content/processed/test/normal")
process_folder(base_path + "/test/defect", "/content/processed/test/defect")

In [7]:
import shutil
os.makedirs("/content/gan_dataset/defects", exist_ok=True)

source = "/content/processed/train/defect"
dest = "/content/gan_dataset/defects"

for file in os.listdir(source):
    shutil.copy(os.path.join(source, file), dest)

In [8]:
import os
print(len(os.listdir("/content/gan_dataset/defects")))

176


In [11]:
# =========================
# 1. IMPORTS
# =========================
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import os

# =========================
# 2. CONFIG
# =========================
image_size = 64
batch_size = 32
latent_dim = 100
num_epochs = 80
lr = 0.0001

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# 3. DATASET PATH
# =========================
dataset_path = "/content/gan_dataset"

# =========================
# 4. DATA LOADING
# =========================
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

dataset = ImageFolder(root=dataset_path, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print("Dataset loaded:", len(dataset))

# =========================
# 5. GENERATOR
# =========================
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 512, 4, 1, 0),
            nn.BatchNorm2d(512),
            nn.ReLU(True),

            nn.ConvTranspose2d(512, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.ConvTranspose2d(64, 3, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

# =========================
# 6. DISCRIMINATOR (FIXED)
# =========================
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1),   # 64 → 32
            nn.LeakyReLU(0.2),

            nn.Conv2d(64, 128, 4, 2, 1), # 32 → 16
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.Conv2d(128, 256, 4, 2, 1), # 16 → 8
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),

            nn.Conv2d(256, 1, 4, 1, 0),  # 8 → 1
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

# =========================
# 7. INITIALIZE
# =========================
G = Generator().to(device)
D = Discriminator().to(device)

criterion = nn.BCELoss()

opt_G = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))

# =========================
# 8. TRAINING LOOP (FIXED)
# =========================
print("Starting GAN training...")

for epoch in range(num_epochs):
    for real, _ in dataloader:

        real = real.to(device)
        batch_size = real.size(0)

        real_labels = torch.ones(batch_size, 1).to(device) * 0.9
        fake_labels = torch.zeros(batch_size, 1).to(device)

        # ---- Train Discriminator ----
        opt_D.zero_grad()

        out_real = D(real)
        out_real = out_real.view(out_real.size(0), -1).mean(dim=1, keepdim=True)
        loss_real = criterion(out_real, real_labels)

        noise = torch.randn(batch_size, latent_dim, 1, 1).to(device)
        fake = G(noise)

        out_fake = D(fake.detach())
        out_fake = out_fake.view(out_fake.size(0), -1).mean(dim=1, keepdim=True)
        loss_fake = criterion(out_fake, fake_labels)

        loss_D = loss_real + loss_fake
        loss_D.backward()
        opt_D.step()

        # ---- Train Generator ----
        opt_G.zero_grad()

        out_fake = D(fake)
        out_fake = out_fake.view(out_fake.size(0), -1).mean(dim=1, keepdim=True)
        loss_G = criterion(out_fake, real_labels)

        loss_G.backward()
        opt_G.step()

        if (epoch+1) % 10 == 0:
              with torch.no_grad():
                  noise = torch.randn(16, latent_dim, 1, 1).to(device)
                  fake = G(noise)

                  os.makedirs("/content/progress", exist_ok=True)
                  torchvision.utils.save_image(
                      fake,
                      f"/content/progress/epoch_{epoch+1}.png",
                      normalize=True
                  )

        print(f"Epoch [{epoch+1}/{num_epochs}]  D Loss: {loss_D.item():.4f}, G Loss: {loss_G.item():.4f}")

        print(f"Epoch [{epoch+1}/{num_epochs}]  D Loss: {loss_D.item():.4f}, G Loss: {loss_G.item():.4f}")

# =========================
# 9. GENERATE IMAGES
# =========================
os.makedirs("/content/generated_defects", exist_ok=True)

G.eval()
with torch.no_grad():
    noise = torch.randn(30, latent_dim, 1, 1).to(device)
    fake_images = G(noise)

    for i, img in enumerate(fake_images):
        torchvision.utils.save_image(img, f"/content/generated_defects/img_{i}.png", normalize=True)

print("✅ Generated images saved in /content/generated_defects/")

Dataset loaded: 176
Starting GAN training...
Epoch [1/80]  D Loss: 1.4082, G Loss: 0.8098
Epoch [1/80]  D Loss: 1.4082, G Loss: 0.8098
Epoch [1/80]  D Loss: 1.4253, G Loss: 0.7729
Epoch [1/80]  D Loss: 1.4253, G Loss: 0.7729
Epoch [1/80]  D Loss: 1.4354, G Loss: 0.7772
Epoch [1/80]  D Loss: 1.4354, G Loss: 0.7772
Epoch [1/80]  D Loss: 1.3966, G Loss: 0.8169
Epoch [1/80]  D Loss: 1.3966, G Loss: 0.8169
Epoch [1/80]  D Loss: 1.3383, G Loss: 0.8874
Epoch [1/80]  D Loss: 1.3383, G Loss: 0.8874
Epoch [1/80]  D Loss: 1.2284, G Loss: 0.9554
Epoch [1/80]  D Loss: 1.2284, G Loss: 0.9554
Epoch [2/80]  D Loss: 1.2039, G Loss: 1.0301
Epoch [2/80]  D Loss: 1.2039, G Loss: 1.0301
Epoch [2/80]  D Loss: 1.1599, G Loss: 1.0720
Epoch [2/80]  D Loss: 1.1599, G Loss: 1.0720
Epoch [2/80]  D Loss: 1.1309, G Loss: 1.1112
Epoch [2/80]  D Loss: 1.1309, G Loss: 1.1112
Epoch [2/80]  D Loss: 1.1507, G Loss: 1.1416
Epoch [2/80]  D Loss: 1.1507, G Loss: 1.1416
Epoch [2/80]  D Loss: 1.0998, G Loss: 1.1823
Epoch [2/8

In [12]:
import shutil

shutil.make_archive('generated_defects', 'zip', '/content/generated_defects')

'/content/generated_defects.zip'

In [13]:
from google.colab import files
files.download('generated_defects.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>